In [1]:
import requests

from datetime import datetime, timezone, timedelta

In [2]:
# Capital -> Região
capitais_regioes = {
    "BRASILIA": "Centro-Oeste",
    "GOIANIA": "Centro-Oeste",
    "CUIABA": "Centro-Oeste",
    "CAMPO GRANDE": "Centro-Oeste",

    "MACEIO": "Nordeste",
    "SALVADOR": "Nordeste",
    "FORTALEZA": "Nordeste",
    "SAO LUIS": "Nordeste",
    "JOAO PESSOA": "Nordeste",
    "RECIFE": "Nordeste",
    "TERESINA": "Nordeste",
    "NATAL": "Nordeste",
    "ARACAJU": "Nordeste",

    "RIO BRANCO": "Norte",
    "MACAPA": "Norte",
    "MANAUS": "Norte",
    "BELEM": "Norte",
    "PORTO VELHO": "Norte",
    "BOA VISTA": "Norte",
    "PALMAS": "Norte",

    "VITORIA": "Sudeste",
    "BELO HORIZONTE": "Sudeste",
    "RIO DE JANEIRO": "Sudeste",
    "SAO PAULO": "Sudeste",

    "CURITIBA": "Sul",
    "PORTO ALEGRE": "Sul",
    "FLORIANOPOLIS": "Sul"
}


In [3]:

url = "https://wis2bra.inmet.gov.br/oapi/collections/stations/items"

todas_estacoes = []

while url:
    resposta = requests.get(url, params={"f": "json", "limit": 1000}, timeout=30)
    resposta.raise_for_status()
    dados = resposta.json()
    todas_estacoes.extend(dados["features"])
    url = next( (link["href"]
                for link in dados.get("links", [])
                if link.get("rel") == "next"),None
              )

estacoes_capitais = {}

for estacao in todas_estacoes:

    prop = estacao["properties"]
    nome = prop["name"].upper()

    for cidade, regiao in capitais_regioes.items():
        if cidade in nome:
            estacoes_capitais[prop["wigos_station_identifier"]] = {"cidade": cidade, "regiao": regiao}

In [4]:

url = ("https://wis2bra.inmet.gov.br/oapi/collections/urn:wmo:md:br-inmet:synop/items")

agora = datetime.now(timezone.utc)
inicio = agora - timedelta(hours=6)

todas_observacoes = []

parametros = {"f": "json", "limit": 1000, "datetime": f"{inicio.isoformat()}/{agora.isoformat()}"}

while url:
    resposta = requests.get(url, params=parametros, timeout=30)
    resposta.raise_for_status()
    dados = resposta.json()
    todas_observacoes.extend(dados["features"])

    url = next( (link["href"]
                for link in dados.get("links", [])
                if link.get("rel") == "next" ), None
              )

    parametros = None

In [ ]:
temperaturas = []

for obs in todas_observacoes:
    prop = obs["properties"]
    wigos = prop["wigos_station_identifier"]
    if (prop["name"] == "air_temperature" and wigos in estacoes_capitais):
        temperaturas.append({
            "regiao": estacoes_capitais[wigos]["regiao"],
            "cidade": estacoes_capitais[wigos]["cidade"],
            "temperatura": prop["value"],
            "dt_referencia": prop["phenomenonTime"]
        })

In [5]:
temperaturas = []

for obs in todas_observacoes:

    prop = obs["properties"]

    # Ignora imediatamente tudo que não for temperatura
    if prop["name"] != "air_temperature":
        continue

    wigos = prop["wigos_station_identifier"]

    # Ignora imediatamente estações que não nos interessam
    if wigos not in estacoes_capitais:
        continue

    capital = estacoes_capitais[wigos]

    temperaturas.append({
        "regiao": capital["regiao"],
        "cidade": capital["cidade"],
        "temperatura": prop["value"],
        "dt_referencia": prop["phenomenonTime"]
    })

In [ ]:
for temperatura in temperaturas:
    print(temperatura)